# 04 - Spending Forecast (LSTM Time Series)

Train an LSTM model to predict the next 7 days of spending from the last 30 days.

**Architecture**: LSTM(32) → Dense(16) → Dense(7)

**Pipeline**:
1. Prepare daily spending time series
2. Create sliding windows (30-day input → 7-day output)
3. Train LSTM forecaster
4. Evaluate (MAE, RMSE, MAPE)
5. Compare with Prophet baseline
6. Export to TFLite for Flutter

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

from evaluate import print_regression_metrics, plot_training_history
from export_tflite import export_keras_to_tflite, verify_tflite_model

print(f'TensorFlow version: {tf.__version__}')

## 1. Load & Prepare Time Series

In [ ]:
# Load daily spending data (preprocessed in notebook 01)
df = pd.read_csv('../data/processed/daily_spending.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Total days: {len(df)}')
print(f'\nDaily spending stats:')
print(df['amount'].describe())

In [ ]:
# Plot the time series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['date'], df['amount'], linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Daily Spending ($)')
ax.set_title('Daily Spending Over Time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fill gaps in the date range (days with no spending → 0)
full_range = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
df_full = pd.DataFrame({'date': full_range})
df_full = df_full.merge(df, on='date', how='left')
df_full['amount'] = df_full['amount'].fillna(0)

print(f'Full date range: {len(df_full)} days (filled gaps with 0)')

# Use amount values as time series
series = df_full['amount'].values.astype(np.float32)
print(f'Series shape: {series.shape}')

## 2. Create Sliding Windows

In [ ]:
# Hyperparameters
LOOKBACK = 30    # Input: last 30 days
HORIZON = 7      # Output: next 7 days
LSTM_UNITS = 32

# Scale to [0, 1]
scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()

print(f'Scaler min: {scaler.data_min_[0]:.2f}')
print(f'Scaler max: {scaler.data_max_[0]:.2f}')

# Create sliding windows
def create_windows(data, lookback, horizon):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback : i + lookback + horizon])
    return np.array(X), np.array(y)

X, y = create_windows(series_scaled, LOOKBACK, HORIZON)
print(f'\nWindows: {X.shape[0]}')
print(f'X shape: {X.shape}  (samples, lookback)')
print(f'y shape: {y.shape}  (samples, horizon)')

In [ ]:
# Train/test split (last 20% as test — no shuffling for time series)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape for LSTM: (samples, timesteps, features)
X_train_lstm = X_train.reshape(-1, LOOKBACK, 1)
X_test_lstm = X_test.reshape(-1, LOOKBACK, 1)

print(f'Train: {X_train_lstm.shape[0]} samples')
print(f'Test:  {X_test_lstm.shape[0]} samples')

## 3. Build & Train LSTM

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(LSTM_UNITS, input_shape=(LOOKBACK, 1)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(HORIZON),  # linear output for regression
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae'],
)

model.summary()

In [ ]:
history = model.fit(
    X_train_lstm, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True, monitor='val_loss'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=5, monitor='val_loss'
        ),
    ],
)

In [ ]:
# Plot training curves
fig = plot_training_history(history, metrics=('mae', 'loss'))
fig.savefig('../models/saved/forecast_training_curves.png', dpi=100)
plt.show()

## 4. Evaluate

In [ ]:
# Predict on test set
y_pred_scaled = model.predict(X_test_lstm)

# Inverse transform to dollar amounts
y_test_dollars = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1, HORIZON)
y_pred_dollars = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(-1, HORIZON)

# Flatten for overall metrics
y_test_flat = y_test_dollars.flatten()
y_pred_flat = y_pred_dollars.flatten()

print('=== Overall Test Metrics ===')
metrics = print_regression_metrics(y_test_flat, y_pred_flat)

# Per-day metrics
print('\n=== Per-Day Metrics ===')
for day in range(HORIZON):
    from sklearn.metrics import mean_absolute_error
    day_mae = mean_absolute_error(y_test_dollars[:, day], y_pred_dollars[:, day])
    print(f'  Day {day+1}: MAE = ${day_mae:.2f}')

In [ ]:
# Plot actual vs predicted for a few test windows
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(y_test_dollars):
        ax.set_visible(False)
        continue
    
    days = np.arange(1, HORIZON + 1)
    ax.bar(days - 0.15, y_test_dollars[i], width=0.3, label='Actual', alpha=0.7)
    ax.bar(days + 0.15, y_pred_dollars[i], width=0.3, label='Predicted', alpha=0.7)
    ax.set_xlabel('Day')
    ax.set_ylabel('$')
    ax.set_title(f'Window {i+1}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('7-Day Forecast: Actual vs Predicted', fontsize=14)
plt.tight_layout()
fig.savefig('../models/saved/forecast_comparison.png', dpi=100)
plt.show()

## 5. Prophet Baseline (Comparison)

In [ ]:
try:
    from prophet import Prophet
    
    # Prepare data for Prophet
    prophet_df = df_full[['date', 'amount']].copy()
    prophet_df.columns = ['ds', 'y']
    
    # Use same train/test split timepoint
    split_date = df_full.iloc[LOOKBACK + split]['date']
    prophet_train = prophet_df[prophet_df['ds'] < split_date]
    prophet_test = prophet_df[prophet_df['ds'] >= split_date]
    
    # Fit Prophet
    prophet_model = Prophet(daily_seasonality=True, yearly_seasonality=False)
    prophet_model.fit(prophet_train)
    
    # Predict
    future = prophet_model.make_future_dataframe(periods=len(prophet_test))
    forecast = prophet_model.predict(future)
    
    prophet_pred = forecast.tail(len(prophet_test))['yhat'].values
    prophet_actual = prophet_test['y'].values[:len(prophet_pred)]
    
    print('=== Prophet Baseline ===')
    prophet_metrics = print_regression_metrics(prophet_actual, prophet_pred)
    
    print(f'\n--- Comparison ---')
    print(f'LSTM MAE:    ${metrics["mae"]:.2f}')
    print(f'Prophet MAE: ${prophet_metrics["mae"]:.2f}')
    print(f'Winner: {"LSTM" if metrics["mae"] <= prophet_metrics["mae"] else "Prophet"}')
    
except ImportError:
    print('Prophet not installed. Skipping baseline comparison.')
    print('Install with: pip install prophet')

## 6. Export to TFLite

In [ ]:
# Save full Keras model
model.save('../models/saved/forecast_keras')
print('Keras model saved.')

# Export TFLite with quantization
tflite_meta = export_keras_to_tflite(
    model,
    output_path='../models/tflite/forecast.tflite',
    quantize=True,
)
print(f'\nTFLite model: {tflite_meta["size_kb"]:.1f} KB')

In [ ]:
import json

# Export forecast config
forecast_config = {
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'lstm_units': LSTM_UNITS,
    'scaler_min': float(scaler.data_min_[0]),
    'scaler_max': float(scaler.data_max_[0]),
    'metrics': {
        'mae': float(metrics['mae']),
        'rmse': float(metrics['rmse']),
        'mape': float(metrics['mape']),
    },
}

with open('../models/tflite/forecast_config.json', 'w') as f:
    json.dump(forecast_config, f, indent=2)
print('Forecast config saved.')
print(f'\nConfig: lookback={LOOKBACK}, horizon={HORIZON}')
print(f'Scaler: min={scaler.data_min_[0]:.2f}, max={scaler.data_max_[0]:.2f}')

In [ ]:
# Verify TFLite model
sample = X_test_lstm[:3].astype(np.float32)
tflite_output = verify_tflite_model('../models/tflite/forecast.tflite', sample)

keras_output = model.predict(sample, verbose=0)

print(f'\nKeras vs TFLite forecast comparison:')
for i in range(3):
    keras_pred = scaler.inverse_transform(keras_output[i].reshape(-1, 1)).flatten()
    tflite_pred = scaler.inverse_transform(tflite_output[i].reshape(-1, 1)).flatten()
    print(f'  Sample {i}:')
    print(f'    Keras:  [{" ".join(f"${v:.0f}" for v in keras_pred)}]')
    print(f'    TFLite: [{" ".join(f"${v:.0f}" for v in tflite_pred)}]')

## Results Summary

### Model Architecture
```
Input(30, 1) → LSTM(32) → Dense(16, relu) → Dropout(0.2) → Dense(7, linear)
```

### Files Exported for Flutter
| File | Description |
|------|-------------|
| `forecast.tflite` | Quantized LSTM forecaster |
| `forecast_config.json` | Lookback/horizon, scaler params, metrics |

### Key Metrics
- Fill in after running: MAE, RMSE, MAPE
- Cold start: requires 30 days of transaction history

### Next: Copy exports to Flutter
```bash
cp ../models/tflite/forecast.tflite ../../app/assets/models/
cp ../models/tflite/forecast_config.json ../../app/assets/models/
```